# Three-Way S / R / P Router — RF and XGB

Same experiment structure as the binary notebooks: `strat_y` and `strat_y_K`, RF/XGB, original-row 80/20 split, no target expansion, exact cross-branch ties excluded/audited, test + entire metrics, confusion matrices, leakage checks, slide tables, and CSV exports.

## Tie handling and scope

The **classifier has exactly three labels: S, R and P**. B is not a router class and is never used as a training or test target.

The router considers only the 18 non-B algorithms in the S, R and P branches. The complete fastest-algorithm set among those 18 is found first. Same-branch ties (for example S+S) are retained as that branch; cross-branch ties (for example S+R) are excluded. No router target expansion is performed.

For important-failure severity only, the denominator remains the **global minimum runtime across all 24 algorithms**, preserving consistency with the previous runtime-oriented analyses. Thus B algorithms can influence the global runtime benchmark, but **never the S/R/P classification target**.


In [2]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier
REPO_ROOT=Path.cwd(); sys.path.insert(0,str(REPO_ROOT)) if str(REPO_ROOT) not in sys.path else None
from restructured_pipeline.src.features import ALL_ALGOS, ROUTER_CLASSB_ALGOS, ROUTER_CLASSS_ALGOS, ROUTER_CLASSR_ALGOS, ROUTER_CLASSP_ALGOS, load_excel_data, process_kcore_list_to_statistics, post_process_data
from restructured_pipeline.src.data_splits import _train_test_split_k_aware
from restructured_pipeline.src.train import load_yaml_config
PROJECT_CONFIG=Path("restructured_pipeline/configs/project.yaml"); MODEL_CONFIG=Path("restructured_pipeline/configs/models_scheme1.yaml")
OUTPUT_DIR=Path("restructured_pipeline/outputs/reports/three_way_SRP_router"); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
project=load_yaml_config(PROJECT_CONFIG); model_cfg=load_yaml_config(MODEL_CONFIG)
FEATURE_COLS=list(project["features"]["feature_cols"]); TEST_SIZE=project["split_rules"]["defaults"]["test_size"]; RANDOM_STATE=project["split_rules"]["defaults"]["random_state"]
CLASS_ORDER=["S","R","P"]
BRANCH_ALGOS={"S":list(ROUTER_CLASSS_ALGOS),"R":list(ROUTER_CLASSR_ALGOS),"P":list(ROUTER_CLASSP_ALGOS)}
SRP_ALGOS=[a for c in CLASS_ORDER for a in BRANCH_ALGOS[c]]
B_ALGOS=list(ROUTER_CLASSB_ALGOS)
assert len(SRP_ALGOS)==18, f"Expected 18 S/R/P algorithms, found {len(SRP_ALGOS)}"
assert len(B_ALGOS)==6, f"Expected 6 B algorithms, found {len(B_ALGOS)}"
assert set(SRP_ALGOS).isdisjoint(B_ALGOS)
assert set(SRP_ALGOS) | set(B_ALGOS) == set(ALL_ALGOS)
print("Router classes:", CLASS_ORDER)
print("Router algorithms:", len(SRP_ALGOS))
for c in CLASS_ORDER: print(c, len(BRANCH_ALGOS[c]), BRANCH_ALGOS[c])

Router classes: ['S', 'R', 'P']
Router algorithms: 18
S 6 ['kPEX-S-bnb', 'kPEX-oriented-S-bnb', 'UMKP-S-bnb', 'UMKP-oriented-S-bnb', 'UMKP+-S-bnb', 'UMKP+-oriented-S-bnb']
R 6 ['kPEX-R-bnb', 'kPEX-oriented-R-bnb', 'UMKP-R-bnb', 'UMKP-oriented-R-bnb', 'UMKP+-R-bnb', 'UMKP+-oriented-R-bnb']
P 6 ['kPEX-P-bnb', 'kPEX-oriented-P-bnb', 'UMKP-P-bnb', 'UMKP-oriented-P-bnb', 'UMKP+-P-bnb', 'UMKP+-oriented-P-bnb']


## 1. Load and preprocess canonical data

In [3]:
d=project["data"]
df=load_excel_data(d["file_path"],timeout_replacement=d["timeout_replacement"],y_cols=d["raw_target_columns"],sep=d["sep"],quotechar=d["quotechar"],engine=d["engine"],rename_columns=d["rename_columns"])
if "kcore_list" in df.columns and any(c not in df.columns for c in FEATURE_COLS): process_kcore_list_to_statistics(df,keep_col=False)
df=post_process_data(df)
missing=[c for c in ["dataset","k_value",*FEATURE_COLS,*ALL_ALGOS] if c not in df.columns]
if missing: raise ValueError(f"Missing columns: {missing}")
print("Original rows:",len(df)); display(df[["dataset","k_value",*FEATURE_COLS]].head())

Original rows: 4279


,dataset,k_value,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize
0,12month1,20,20,52328,1585220,30.294,9214,57,19,39,16785.102564,52328,1030,14354.722410
1,130bit,8,8,584,6059,10.375,322,12,8,5,564.400000,584,509,28.443628
2,130bit,9,9,584,6059,10.375,322,12,8,5,564.400000,584,509,28.443628
3,130bit,10,10,583,6051,10.379,321,12,9,4,559.500000,583,509,29.853810
4,144,4,4,144319,1072349,7.430,26,9,7,3,140266.333333,144319,132943,5188.210310


## 2. Build S/R/P target and exclude exact cross-branch ties
A tie within one branch is harmless. A row is excluded only when two or more branches share the exact minimum branch runtime.

In [19]:
records = []

# All four router classes.
ALL_BRANCH_ALGOS = {
    "B": list(ROUTER_CLASSB_ALGOS),
    "S": list(ROUTER_CLASSS_ALGOS),
    "R": list(ROUTER_CLASSR_ALGOS),
    "P": list(ROUTER_CLASSP_ALGOS),
}

CLASS_ORDER_4 = ["B", "S", "R", "P"]

for idx, row in df.iterrows():

    # ---------------------------------------------------------
    # Find the true global optimum across ALL 24 algorithms.
    # ---------------------------------------------------------
    runtimes = {
        algo: float(row[algo])
        for algo in ALL_ALGOS
    }

    best = min(runtimes.values())

    # Keep every algorithm tied at the global minimum.
    fastest_algorithms = [
        algo
        for algo in ALL_ALGOS
        if runtimes[algo] == best
    ]

    # ---------------------------------------------------------
    # Map globally fastest algorithms to B / S / R / P.
    # ---------------------------------------------------------
    mapped_classes = sorted({
        branch
        for branch, algos in ALL_BRANCH_ALGOS.items()
        if any(
            algo in algos
            for algo in fastest_algorithms
        )
    })

    same_class_tie = (
        len(fastest_algorithms) > 1
        and len(mapped_classes) == 1
    )

    cross_class_tie = (
        len(mapped_classes) > 1
    )

    records.append({
        "source_row_index": idx,
        "dataset": row["dataset"],
        "k_value": row["k_value"],

        "global_fastest_runtime": best,

        "global_fastest_algorithms":
            " | ".join(fastest_algorithms),

        "mapped_classes":
            " | ".join(mapped_classes),

        "number_of_fastest_algorithms":
            len(fastest_algorithms),

        "tie_category": (
            "Cross-Class Tie (Excluded)"
            if cross_class_tie
            else "Same-Class Tie (Kept)"
            if same_class_tie
            else "No Tie"
        ),

        "target": (
            mapped_classes[0]
            if len(mapped_classes) == 1
            else pd.NA
        ),
    })


# ============================================================
# Audit
# ============================================================

target_audit = pd.DataFrame(records)

tie_audit = target_audit[
    target_audit["tie_category"].eq(
        "Cross-Class Tie (Excluded)"
    )
].copy()

same_class_ties = target_audit[
    target_audit["tie_category"].eq(
        "Same-Class Tie (Kept)"
    )
].copy()

usable = target_audit[
    target_audit["target"].notna()
].copy()


# ============================================================
# B / S / R / P counts
# ============================================================

class_counts = (
    usable["target"]
    .value_counts()
    .reindex(CLASS_ORDER_4, fill_value=0)
)

audit_summary = pd.DataFrame([{
    "Total Original Cases":
        len(df),

    "Final Cases Used":
        len(usable),

    "Same-Class Ties Kept":
        len(same_class_ties),

    "Cross-Class Ties Excluded":
        len(tie_audit),

    "Cross-Class Tie Rate (%)":
        round(
            100 * len(tie_audit) / len(df),
            2,
        )
        if len(df)
        else 0,

    "B Cases":
        int(class_counts["B"]),

    "S Cases":
        int(class_counts["S"]),

    "R Cases":
        int(class_counts["R"]),

    "P Cases":
        int(class_counts["P"]),
}])


display(audit_summary)

print("\nGlobal-fastest branch distribution:")
display(
    class_counts
    .rename("Cases")
    .to_frame()
)

print("\nCross-class ties:")
display(
    tie_audit.head(20)
)


# ============================================================
# Save audit
# ============================================================

audit_summary.to_csv(
    OUTPUT_DIR / "target_audit_summary.csv",
    index=False,
)

target_audit.to_csv(
    OUTPUT_DIR / "target_audit_details.csv",
    index=False,
)

tie_audit.to_csv(
    OUTPUT_DIR / "cross_class_ties_excluded.csv",
    index=False,
)

,Total Original Cases,Final Cases Used,Same-Class Ties Kept,Cross-Class Ties Excluded,Cross-Class Tie Rate (%),B Cases,S Cases,R Cases,P Cases
0,4279,3767,15,512,11.97,1560,765,360,1082



Global-fastest branch distribution:


,Cases
target,
B,1560
S,765
R,360
P,1082



Cross-class ties:


,source_row_index,dataset,k_value,global_fastest_runtime,global_fastest_algorithms,mapped_classes,number_of_fastest_algorithms,tie_category,target
65,65,actor-collaboration,6,0.0011,kPEX-oriented-S-bnb | kPEX-oriented-R-bnb | UM...,B | R | S,3,Cross-Class Tie (Excluded),<NA>
66,66,actor-collaboration,7,0.0014,kPEX-oriented-S-bnb | kPEX-oriented-R-bnb | kP...,B | P | R | S,11,Cross-Class Tie (Excluded),<NA>
67,67,actor-collaboration,8,0.0016,kPEX-oriented-S-bnb | kPEX-oriented-P-bnb | UM...,B | P | S,4,Cross-Class Tie (Excluded),<NA>
68,68,actor-collaboration,9,0.0018,kPEX-oriented-R-bnb | UMKP-oriented-B-bnb,B | R,2,Cross-Class Tie (Excluded),<NA>
69,69,actor-collaboration,10,0.0021,kPEX-oriented-S-bnb | kPEX-oriented-P-bnb | kP...,B | P | S,5,Cross-Class Tie (Excluded),<NA>
135,135,af_5_k101,6,0.2916,kPEX-oriented-S-bnb | UMKP+-oriented-B-bnb,B | S,2,Cross-Class Tie (Excluded),<NA>
163,163,aff-wiki-wordbypage,3,0.0035,UMKP+-oriented-S-bnb | UMKP+-oriented-R-bnb | ...,B | P | R | S,4,Cross-Class Tie (Excluded),<NA>
164,164,aff-wiki-wordbypage,4,0.0035,UMKP-oriented-S-bnb | UMKP-oriented-R-bnb | UM...,P | R | S,5,Cross-Class Tie (Excluded),<NA>
167,167,aff-wiki-wordbypage,7,0.0000,kPEX-S-bnb | kPEX-R-bnb | kPEX-P-bnb | kPEX-B-...,B | P | R | S,24,Cross-Class Tie (Excluded),<NA>
168,168,aff-wiki-wordbypage,8,0.0000,kPEX-S-bnb | kPEX-R-bnb | kPEX-P-bnb | kPEX-B-...,B | P | R | S,24,Cross-Class Tie (Excluded),<NA>


In [ ]:
records=[]
for idx,row in df.iterrows():
    # Target construction uses ONLY the 18 S/R/P algorithms.
    runtimes={a:float(row[a]) for a in ALL_ALGOS}
    best=min(runtimes.values())
    fastest_algorithms=[a for a in SRP_ALGOS if runtimes[a]==best]

    mapped_classes=sorted({
        c for c,algos in BRANCH_ALGOS.items()
        if any(a in algos for a in fastest_algorithms)
    })

    same_class_tie=(len(fastest_algorithms)>1 and len(mapped_classes)==1)
    cross_class_tie=(len(mapped_classes)>1)

    records.append({
        "source_row_index":idx,
        "dataset":row["dataset"],
        "k_value":row["k_value"],
        "fastest_runtime_among_18":best,
        "fastest_algorithms_among_18":" | ".join(fastest_algorithms),
        "mapped_router_classes":" | ".join(mapped_classes),
        "number_of_fastest_algorithms":len(fastest_algorithms),
        "tie_category":(
            "Cross-Class Tie (Excluded)" if cross_class_tie
            else "Same-Class Tie (Kept)" if same_class_tie
            else "No Tie"
        ),
        "target":mapped_classes[0] if len(mapped_classes)==1 else pd.NA,
    })

target_audit=pd.DataFrame(records)
tie_audit=target_audit[target_audit["tie_category"].eq("Cross-Class Tie (Excluded)")].copy()
same_class_ties=target_audit[target_audit["tie_category"].eq("Same-Class Tie (Kept)")].copy()
usable=target_audit[target_audit.target.notna()].copy()

work=df.loc[usable.source_row_index].copy()
work["target"]=usable.set_index("source_row_index").loc[work.index,"target"]

# Hard guard: B can never enter this classifier.
unexpected=set(work["target"].dropna().unique())-set(CLASS_ORDER)
if unexpected:
    raise ValueError(f"Unexpected router labels: {sorted(unexpected)}")
if work["target"].eq("B").any():
    raise ValueError("B label entered the S/R/P router target.")

audit_summary=pd.DataFrame([{
    "Total Original Cases":len(df),
    "Final Cases Used":len(work),
    "Same-Class Ties Kept":len(same_class_ties),
    "Cross-Class Ties Excluded":len(tie_audit),
    "Cross-Class Tie Rate (%)":round(100*len(tie_audit)/len(df),2) if len(df) else 0,
    "B Cases":int(work.target.eq("B").sum()),
    "S Cases":int(work.target.eq("S").sum()),
    "R Cases":int(work.target.eq("R").sum()),
    "P Cases":int(work.target.eq("P").sum()),
}])

display(audit_summary)
display(tie_audit.head(20))
audit_summary.to_csv(OUTPUT_DIR/"target_audit_summary.csv",index=False)
target_audit.to_csv(OUTPUT_DIR/"target_audit_details.csv",index=False)
tie_audit.to_csv(OUTPUT_DIR/"cross_class_ties_excluded.csv",index=False)


,Total Original Cases,Final Cases Used,Same-Class Ties Kept,Cross-Class Ties Excluded,Cross-Class Tie Rate (%),B Cases,S Cases,R Cases,P Cases
0,4279,2276,74,443,10.35,0,775,377,1124


,source_row_index,dataset,k_value,fastest_runtime_among_18,fastest_algorithms_among_18,mapped_router_classes,number_of_fastest_algorithms,tie_category,target
65,65,actor-collaboration,6,0.0011,kPEX-oriented-S-bnb | kPEX-oriented-R-bnb | UM...,R | S,3,Cross-Class Tie (Excluded),<NA>
66,66,actor-collaboration,7,0.0014,kPEX-oriented-S-bnb | kPEX-oriented-R-bnb | kP...,P | R | S,11,Cross-Class Tie (Excluded),<NA>
67,67,actor-collaboration,8,0.0016,kPEX-oriented-S-bnb | kPEX-oriented-P-bnb | UM...,P | S,4,Cross-Class Tie (Excluded),<NA>
69,69,actor-collaboration,10,0.0021,kPEX-oriented-S-bnb | kPEX-oriented-P-bnb | kP...,P | S,5,Cross-Class Tie (Excluded),<NA>
163,163,aff-wiki-wordbypage,3,0.0035,UMKP+-oriented-S-bnb | UMKP+-oriented-R-bnb | ...,P | R | S,4,Cross-Class Tie (Excluded),<NA>
164,164,aff-wiki-wordbypage,4,0.0035,UMKP-oriented-S-bnb | UMKP-oriented-R-bnb | UM...,P | R | S,5,Cross-Class Tie (Excluded),<NA>
167,167,aff-wiki-wordbypage,7,0.0000,kPEX-S-bnb | kPEX-R-bnb | kPEX-P-bnb | kPEX-B-...,P | R | S,24,Cross-Class Tie (Excluded),<NA>
168,168,aff-wiki-wordbypage,8,0.0000,kPEX-S-bnb | kPEX-R-bnb | kPEX-P-bnb | kPEX-B-...,P | R | S,24,Cross-Class Tie (Excluded),<NA>
169,169,aff-wiki-wordbypage,9,0.0000,kPEX-S-bnb | kPEX-R-bnb | kPEX-P-bnb | kPEX-B-...,P | R | S,24,Cross-Class Tie (Excluded),<NA>
170,170,aff-wiki-wordbypage,10,0.0000,kPEX-S-bnb | kPEX-R-bnb | kPEX-P-bnb | kPEX-B-...,P | R | S,24,Cross-Class Tie (Excluded),<NA>


## 3. Original-row `strat_y` and `strat_y_K` splitting — no target expansion

In [5]:
def make_split(frame,mode):
    X=frame[FEATURE_COLS].copy(); y=frame.target.copy()
    if mode=="strat_y": Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=TEST_SIZE,random_state=RANDOM_STATE,stratify=y)
    else:
        labels=y.astype(str)+"_k"+frame.k_value.astype(str)
        Xtr,Xte,ytr,yte=_train_test_split_k_aware(X,y,test_size=TEST_SIZE,random_state=RANDOM_STATE,stratify_labels=labels,context_label="three_way_SRP_strat_y_K")
    if set(Xtr.index)&set(Xte.index): raise ValueError("Train/test leakage")
    if Xte.index.duplicated().any(): raise ValueError("Duplicate test indices")
    return Xtr,Xte,ytr,yte
splits={m:make_split(work,m) for m in ("strat_y","strat_y_K")}
rows=[]
for m,(Xtr,Xte,ytr,yte) in splits.items(): rows.append({"split_mode":m,"train_rows":len(Xtr),"test_rows":len(Xte),"overlap":len(set(Xtr.index)&set(Xte.index)),"duplicate_test_indices":int(Xte.index.duplicated().sum()),**{f"train_{c}":int(ytr.eq(c).sum()) for c in CLASS_ORDER},**{f"test_{c}":int(yte.eq(c).sum()) for c in CLASS_ORDER}})
split_audit=pd.DataFrame(rows); display(split_audit); split_audit.to_csv(OUTPUT_DIR/"split_audit.csv",index=False)

three_way_SRP_strat_y_K: forced 0 row(s) into training before the K-aware split.
three_way_SRP_strat_y_K: completed true target+K stratification on 33 combined class(es).
three_way_SRP_strat_y_K: final train/test sizes = 3032/759.


,split_mode,train_rows,test_rows,overlap,duplicate_test_indices,train_S,train_R,train_P,test_S,test_R,test_P
0,strat_y,3032,759,0,0,814,472,1746,204,118,437
1,strat_y_K,3032,759,0,0,815,472,1745,203,118,438


## 4. RF/XGB configuration

In [6]:
def cfg(family,mode):
    prefix="rf" if family=="random_forest" else "xgb"; return dict(model_cfg["scheme1"][family]["variants"][f"{prefix}_{mode}"]["hyperparameters"])
def build_rf(mode): return RandomForestClassifier(**cfg("random_forest",mode))
def build_xgb(mode):
    p=cfg("xgboost",mode); p.update(objective="multi:softprob",num_class=3); return XGBClassifier(**p)
le=LabelEncoder(); le.fit(CLASS_ORDER); print(dict(zip(le.classes_,le.transform(le.classes_))))

{'P': 0, 'R': 1, 'S': 2}


## 5. Metrics and confusion matrices

In [7]:
def metric_row(y,p,mode,model,scope):
    pm,rm,fm,_=precision_recall_fscore_support(y,p,labels=CLASS_ORDER,average="macro",zero_division=0); pw,rw,fw,_=precision_recall_fscore_support(y,p,labels=CLASS_ORDER,average="weighted",zero_division=0); pc,rc,fc,sc=precision_recall_fscore_support(y,p,labels=CLASS_ORDER,average=None,zero_division=0)
    z={"split_mode":mode,"model":model,"scope":scope,"rows":len(y),"accuracy":accuracy_score(y,p),"macro_precision":pm,"macro_recall":rm,"macro_f1":fm,"weighted_precision":pw,"weighted_recall":rw,"weighted_f1":fw}
    for i,c in enumerate(CLASS_ORDER): z.update({f"{c}_precision":pc[i],f"{c}_recall":rc[i],f"{c}_f1":fc[i],f"{c}_support":int(sc[i])})
    return z
def plot_cm(y,p,mode,model,scope):
    cm=confusion_matrix(y,p,labels=CLASS_ORDER); fig,ax=plt.subplots(figsize=(6,5)); ConfusionMatrixDisplay(cm,display_labels=CLASS_ORDER).plot(ax=ax,values_format="d"); ax.set_title(f"S/R/P - {model} - {mode} - {scope}"); fig.tight_layout(); fig.savefig(OUTPUT_DIR/f"confusion_{model}_{mode}_{scope}.png",dpi=180,bbox_inches="tight"); plt.show()

## 6. Train four models; evaluate held-out test and entire usable data

In [8]:
metric_rows=[]; pred_frames=[]
for mode,(Xtr,Xte,ytr,yte) in splits.items():
    for name in ("RF","XGB"):
        if name=="RF": model=build_rf(mode); model.fit(Xtr,ytr); predict=lambda X:model.predict(X)
        else: model=build_xgb(mode); model.fit(Xtr,le.transform(ytr)); predict=lambda X:le.inverse_transform(model.predict(X).astype(int))
        for scope,Xe,ye in (("test_data",Xte,yte),("entire_data",work[FEATURE_COLS],work.target)):
            yp=predict(Xe); metric_rows.append(metric_row(ye,yp,mode,name,scope)); plot_cm(ye,yp,mode,name,scope)
            q=work.loc[Xe.index,["dataset","k_value"]].copy(); q["source_row_index"]=Xe.index; q["split_mode"]=mode; q["model"]=name; q["scope"]=scope; q["true_target"]=np.asarray(ye); q["predicted_target"]=np.asarray(yp); q["correct"]=q.true_target.eq(q.predicted_target); pred_frames.append(q)
metrics=pd.DataFrame(metric_rows); predictions=pd.concat(pred_frames,ignore_index=True); display(metrics.round(4)); metrics.to_csv(OUTPUT_DIR/"metrics_all.csv",index=False); predictions.to_csv(OUTPUT_DIR/"predictions_all.csv",index=False)
for mode,(Xtr,Xte,_,_) in splits.items(): pd.DataFrame({"source_row_index":Xtr.index}).to_csv(OUTPUT_DIR/f"train_indices_{mode}.csv",index=False); pd.DataFrame({"source_row_index":Xte.index}).to_csv(OUTPUT_DIR/f"test_indices_{mode}.csv",index=False)

C:\Users\n1a9m\AppData\Local\Temp\ipykernel_25004\1045477391.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  cm=confusion_matrix(y,p,labels=CLASS_ORDER); fig,ax=plt.subplots(figsize=(6,5)); ConfusionMatrixDisplay(cm,display_labels=CLASS_ORDER).plot(ax=ax,values_format="d"); ax.set_title(f"S/R/P - {model} - {mode} - {scope}"); fig.tight_layout(); fig.savefig(OUTPUT_DIR/f"confusion_{model}_{mode}_{scope}.png",dpi=180,bbox_inches="tight"); plt.show()
C:\Users\n1a9m\AppData\Local\Temp\ipykernel_25004\1045477391.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  cm=confusion_matrix(y,p,labels=CLASS_ORDER); fig,ax=plt.subplots(figsize=(6,5)); ConfusionMatrixDisplay(cm,display_labels=CLASS_ORDER).plot(ax=ax,values_format="d"); ax.set_title(f"S/R/P - {model} - {mode} - {scope}"); fig.tight_layout(); fig.savefig(OUTPUT_DIR/f"confusion_{model}_{mode}_{scope}.png",dpi=180,bbox_inches="tight"); plt.show()
c:\Users\n1a9m\Anaconda3_\Li

,split_mode,model,scope,rows,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,...,S_f1,S_support,R_precision,R_recall,R_f1,R_support,P_precision,P_recall,P_f1,P_support
0,strat_y,RF,test_data,759,0.6772,0.5941,0.5839,0.5879,0.6721,0.6772,...,0.5707,204,0.3950,0.3983,0.3966,118,0.7749,0.8192,0.7964,437
1,strat_y,RF,entire_data,3791,0.8892,0.8557,0.8766,0.8651,0.8927,0.8892,...,0.8668,1018,0.7590,0.8542,0.8038,590,0.9408,0.9093,0.9248,2183
2,strat_y,XGB,test_data,759,0.6891,0.6048,0.5866,0.5941,0.6775,0.6891,...,0.5964,204,0.4158,0.3559,0.3836,118,0.7717,0.8352,0.8022,437
3,strat_y,XGB,entire_data,3791,0.9209,0.9060,0.8940,0.8998,0.9203,0.9209,...,0.9043,1018,0.8670,0.8288,0.8475,590,0.9381,0.9574,0.9476,2183
4,strat_y_K,RF,test_data,759,0.6548,0.5578,0.5497,0.5528,0.6457,0.6548,...,0.5281,203,0.3737,0.3136,0.3410,118,0.7753,0.8037,0.7892,438
5,strat_y_K,RF,entire_data,3791,0.9111,0.8849,0.8925,0.8886,0.9119,0.9111,...,0.8854,1018,0.8195,0.8542,0.8365,590,0.9486,0.9391,0.9438,2183
6,strat_y_K,XGB,test_data,759,0.6772,0.5916,0.5774,0.5831,0.6667,0.6772,...,0.5636,203,0.4286,0.3559,0.3889,118,0.7754,0.8196,0.7969,438
7,strat_y_K,XGB,entire_data,3791,0.9159,0.8988,0.8899,0.8942,0.9153,0.9159,...,0.8929,1018,0.8609,0.8288,0.8446,590,0.9386,0.9519,0.9452,2183


## 7. Slide-ready held-out table
Use this as the main result table.

In [9]:
cols=["split_mode","model","accuracy","macro_f1","weighted_f1"]+[f"{c}_{m}" for c in CLASS_ORDER for m in ("precision","recall","f1")]
slide=metrics.loc[metrics.scope.eq("test_data"),cols].copy(); nums=slide.select_dtypes(include="number").columns; slide[nums]=(100*slide[nums]).round(2); display(slide); slide.to_csv(OUTPUT_DIR/"slide_table_test_metrics.csv",index=False)

,split_mode,model,accuracy,macro_f1,weighted_f1,S_precision,S_recall,S_f1,R_precision,R_recall,R_f1,P_precision,P_recall,P_f1
0,strat_y,RF,67.72,58.79,67.36,61.24,53.43,57.07,39.50,39.83,39.66,77.49,81.92,79.64
2,strat_y,XGB,68.91,59.41,68.18,62.70,56.86,59.64,41.58,35.59,38.36,77.17,83.52,80.22
4,strat_y_K,RF,65.48,55.28,64.97,52.43,53.20,52.81,37.37,31.36,34.10,77.53,80.37,78.92
6,strat_y_K,XGB,67.72,58.31,67.11,57.07,55.67,56.36,42.86,35.59,38.89,77.54,81.96,79.69


## 8. Slide-ready test vs entire table
Entire-data metrics include training rows and should be presented as descriptive fit metrics.

In [10]:
compact=metrics[["split_mode","model","scope","accuracy","macro_f1","weighted_f1"]].copy(); compact[["accuracy","macro_f1","weighted_f1"]]=(100*compact[["accuracy","macro_f1","weighted_f1"]]).round(2); display(compact); compact.to_csv(OUTPUT_DIR/"slide_table_test_vs_entire.csv",index=False)

,split_mode,model,scope,accuracy,macro_f1,weighted_f1
0,strat_y,RF,test_data,67.72,58.79,67.36
1,strat_y,RF,entire_data,88.92,86.51,89.04
2,strat_y,XGB,test_data,68.91,59.41,68.18
3,strat_y,XGB,entire_data,92.09,89.98,92.04
4,strat_y_K,RF,test_data,65.48,55.28,64.97
5,strat_y_K,RF,entire_data,91.11,88.86,91.14
6,strat_y_K,XGB,test_data,67.72,58.31,67.11
7,strat_y_K,XGB,entire_data,91.59,89.42,91.55


## 9. Generalization-gap table

In [11]:
gap=metrics.pivot(index=["split_mode","model"],columns="scope",values=["accuracy","weighted_f1"]); gap.columns=["_".join(c) for c in gap.columns]; gap=gap.reset_index(); gap["accuracy_gap_entire_minus_test"]=gap.accuracy_entire_data-gap.accuracy_test_data; gap["weighted_f1_gap_entire_minus_test"]=gap.weighted_f1_entire_data-gap.weighted_f1_test_data
for c in [x for x in gap if "accuracy" in x or "weighted_f1" in x]: gap[c]=(100*gap[c]).round(2)
display(gap); gap.to_csv(OUTPUT_DIR/"slide_table_generalization_gap.csv",index=False)

,split_mode,model,accuracy_entire_data,accuracy_test_data,weighted_f1_entire_data,weighted_f1_test_data,accuracy_gap_entire_minus_test,weighted_f1_gap_entire_minus_test
0,strat_y,RF,88.92,67.72,89.04,67.36,21.20,21.68
1,strat_y,XGB,92.09,68.91,92.04,68.18,23.18,23.86
2,strat_y_K,RF,91.11,65.48,91.14,64.97,25.63,26.17
3,strat_y_K,XGB,91.59,67.72,91.55,67.11,23.86,24.44


## Slide interpretation checklist
Prioritize held-out Accuracy, Weighted F1, Macro F1, S/R/P recall and the held-out confusion matrix. Compare RF vs XGB and `strat_y` vs `strat_y_K`. Report excluded exact cross-branch ties. Treat entire-data metrics as fit diagnostics because they include training rows.

## Wrong predictions and important routing failures

This section separates **classification errors** from **runtime-important routing errors**.

For each prediction we report the number of evaluated datapoints, correct predictions, wrong predictions, and wrong-prediction rate. A wrong routing decision is counted as an **important failure** only when:

1. the true global-optimum runtime is **greater than 5 seconds**, and
2. the best runtime available inside the **predicted branch/family** is **greater than 1.2× the global optimum**.

This evaluates the runtime consequence of the router decision. It does **not** assume that the downstream within-family algorithm selector necessarily chooses that family's optimum; therefore it is an optimistic routing-level failure analysis.


In [12]:
IMPORTANT_RUNTIME_SECONDS = 5.0
IMPORTANT_RATIO_THRESHOLD = 1.20

analysis_predictions = predictions.rename(columns={
    "scope": "evaluation_scope",
    "true_target": "true_label",
    "predicted_target": "predicted_label",
}).copy()
analysis_predictions["true_label"] = analysis_predictions["true_label"].astype(str)
analysis_predictions["predicted_label"] = analysis_predictions["predicted_label"].astype(str)

runtime_source = work

def family_best_runtime(row, label):
    if label not in BRANCH_ALGOS:
        raise ValueError(f"Unknown S/R/P label: {label!r}")
    return float(row[BRANCH_ALGOS[label]].min())

def attach_runtime_consequence(r):
    src = runtime_source.loc[int(r["source_row_index"])]
    global_best = float(src[list(ALL_ALGOS)].min())
    predicted_best = family_best_runtime(src, r["predicted_label"])
    return pd.Series({"global_fastest_runtime": global_best, "predicted_branch_best_runtime": predicted_best})

analysis_predictions[["global_fastest_runtime", "predicted_branch_best_runtime"]] = analysis_predictions.apply(attach_runtime_consequence, axis=1)

analysis_predictions["wrong_prediction"] = analysis_predictions["true_label"].ne(analysis_predictions["predicted_label"])
analysis_predictions["routing_time_ratio"] = analysis_predictions["predicted_branch_best_runtime"] / analysis_predictions["global_fastest_runtime"]
analysis_predictions["important_failure"] = (
    analysis_predictions["wrong_prediction"]
    & analysis_predictions["global_fastest_runtime"].gt(IMPORTANT_RUNTIME_SECONDS)
    & analysis_predictions["routing_time_ratio"].gt(IMPORTANT_RATIO_THRESHOLD)
)

# Table 1: prediction population and ordinary classification errors.
wrong_prediction_table = (
    analysis_predictions.groupby(["split_mode", "model", "evaluation_scope"], dropna=False)
    .agg(
        total_datapoints=("source_row_index", "size"),
        correct_predictions=("wrong_prediction", lambda s: int((~s).sum())),
        wrong_predictions=("wrong_prediction", "sum"),
    )
    .reset_index()
)
wrong_prediction_table["wrong_predictions"] = wrong_prediction_table["wrong_predictions"].astype(int)
wrong_prediction_table["accuracy_percent"] = (
    100 * wrong_prediction_table["correct_predictions"] / wrong_prediction_table["total_datapoints"]
).round(2)
wrong_prediction_table["wrong_prediction_percent"] = (
    100 * wrong_prediction_table["wrong_predictions"] / wrong_prediction_table["total_datapoints"]
).round(2)

# Table 2: consequential subset of the wrong predictions.
important_failure_table = (
    analysis_predictions.groupby(["split_mode", "model", "evaluation_scope"], dropna=False)
    .agg(
        total_datapoints=("source_row_index", "size"),
        wrong_predictions=("wrong_prediction", "sum"),
        important_failures=("important_failure", "sum"),
    )
    .reset_index()
)
important_failure_table[["wrong_predictions", "important_failures"]] = important_failure_table[["wrong_predictions", "important_failures"]].astype(int)
important_failure_table["important_failures_percent_of_wrong"] = np.where(
    important_failure_table["wrong_predictions"].gt(0),
    100 * important_failure_table["important_failures"] / important_failure_table["wrong_predictions"],
    0.0,
).round(2)
important_failure_table["important_failures_percent_of_all"] = (
    100 * important_failure_table["important_failures"] / important_failure_table["total_datapoints"]
).round(2)

print("Wrong-prediction analysis")
display(wrong_prediction_table)
print("Important routing failures (>5 s global optimum and >1.2x routing ratio)")
display(important_failure_table)

wrong_prediction_table.to_csv(OUTPUT_DIR / "slide_table_wrong_predictions.csv", index=False)
important_failure_table.to_csv(OUTPUT_DIR / "slide_table_important_failures.csv", index=False)
analysis_predictions.to_csv(OUTPUT_DIR / "prediction_failure_analysis.csv", index=False)


Wrong-prediction analysis


,split_mode,model,evaluation_scope,total_datapoints,correct_predictions,wrong_predictions,accuracy_percent,wrong_prediction_percent
0,strat_y,RF,entire_data,3791,3371,420,88.92,11.08
1,strat_y,RF,test_data,759,514,245,67.72,32.28
2,strat_y,XGB,entire_data,3791,3491,300,92.09,7.91
3,strat_y,XGB,test_data,759,523,236,68.91,31.09
4,strat_y_K,RF,entire_data,3791,3454,337,91.11,8.89
5,strat_y_K,RF,test_data,759,497,262,65.48,34.52
6,strat_y_K,XGB,entire_data,3791,3472,319,91.59,8.41
7,strat_y_K,XGB,test_data,759,514,245,67.72,32.28


Important routing failures (>5 s global optimum and >1.2x routing ratio)


,split_mode,model,evaluation_scope,total_datapoints,wrong_predictions,important_failures,important_failures_percent_of_wrong,important_failures_percent_of_all
0,strat_y,RF,entire_data,3791,420,18,4.29,0.47
1,strat_y,RF,test_data,759,245,13,5.31,1.71
2,strat_y,XGB,entire_data,3791,300,12,4.00,0.32
3,strat_y,XGB,test_data,759,236,12,5.08,1.58
4,strat_y_K,RF,entire_data,3791,337,16,4.75,0.42
5,strat_y_K,RF,test_data,759,262,16,6.11,2.11
6,strat_y_K,XGB,entire_data,3791,319,17,5.33,0.45
7,strat_y_K,XGB,test_data,759,245,16,6.53,2.11


## Wrong predictions and important failures

Use the existing repository `filter_tolerance_sheet_for_important_failures()` function. Eight detail groups are produced: 2 splits × 2 models × held-out/entire.

**Consistency rule:** the S/R/P router target is defined using only the 18 S/R/P algorithms, but important-failure runtime severity is measured against the **global minimum runtime across all 24 algorithms**, exactly as in the previous runtime-oriented failure analyses.


In [14]:
from restructured_pipeline.src.compare_wrong_predictions import filter_tolerance_sheet_for_important_failures

def _best_branch_runtime(source_row, branch):
    algos = BRANCH_ALGOS[branch]
    vals = [(float(source_row[a]), a) for a in algos]
    best = min(v[0] for v in vals)
    return best, [a for rt,a in vals if rt == best]

def _global_24_fastest_runtime(source_row):
    """Return the true global optimum across all 24 algorithms."""
    vals = [(float(source_row[a]), a) for a in ALL_ALGOS]
    best = min(v[0] for v in vals)
    return best, [a for rt,a in vals if rt == best]

failure_detail = {}
failure_summary_rows = []
for (split_mode, model_name, scope), group in predictions.groupby(["split_mode","model","scope"], sort=False):
    enriched = group.copy()
    fastest_times=[]; fastest_algos=[]; predicted_times=[]; predicted_algos=[]
    for _,r in enriched.iterrows():
        src = work.loc[int(r["source_row_index"])]
        ft,fa = _global_24_fastest_runtime(src)
        pt,pa = _best_branch_runtime(src, r["predicted_target"])
        fastest_times.append(ft); fastest_algos.append(" | ".join(fa)); predicted_times.append(pt); predicted_algos.append(" | ".join(pa))
    enriched["fastest_time"] = fastest_times
    # enriched["global_fastest_time"] = fastest_times
    enriched["fastest_algorithm"] = fastest_algos
    # enriched["global_fastest_algorithms"] = fastest_algos
    enriched["routing_time_ratio"] = predicted_times/ enriched["fastest_time"]
    enriched["best_algorithms_in_predicted_branch"] = predicted_algos
    enriched["predicted_time"] = predicted_times
    
    wrong = enriched.loc[~enriched["correct"]].copy()
    important, stats = filter_tolerance_sheet_for_important_failures(wrong)
    failure_detail[(split_mode,model_name,scope)] = important
    failure_summary_rows.append({
        "Stratification": "Target only" if split_mode=="strat_y" else "Target + K",
        "Model": model_name, "Evaluation Scope": scope, "Total Predictions": len(enriched),
        "Wrong Predictions": len(wrong), "Important Failures": len(important),
        "Important / Wrong (%)": round(100*len(important)/len(wrong),2) if len(wrong) else 0,
        "Important / All (%)": round(100*len(important)/len(enriched),2) if len(enriched) else 0,
    })
failure_summary = pd.DataFrame(failure_summary_rows)
display(failure_summary)
failure_summary.to_csv(OUTPUT_DIR / "wrong_and_important_failure_summary.csv", index=False)


,Stratification,Model,Evaluation Scope,Total Predictions,Wrong Predictions,Important Failures,Important / Wrong (%),Important / All (%)
0,Target only,RF,test_data,759,245,20,8.16,2.64
1,Target only,RF,entire_data,3791,420,25,5.95,0.66
2,Target only,XGB,test_data,759,236,17,7.20,2.24
3,Target only,XGB,entire_data,3791,300,17,5.67,0.45
4,Target + K,RF,test_data,759,262,20,7.63,2.64
5,Target + K,RF,entire_data,3791,337,21,6.23,0.55
6,Target + K,XGB,test_data,759,245,20,8.16,2.64
7,Target + K,XGB,entire_data,3791,319,22,6.90,0.58


## Important-failure workbook and held-out/entire overlap

The workbook contains a summary plus eight detail sheets. The overlap table counts the same `(dataset, K)` important failure appearing in held-out and entire-data evaluation for a given split/model.


In [15]:
with pd.ExcelWriter(OUTPUT_DIR / "important_failure_cases.xlsx", engine="openpyxl") as writer:
    failure_summary.to_excel(writer, sheet_name="summary", index=False)
    for (split_mode,model_name,scope),frame in failure_detail.items():
        short = "test" if scope=="test_data" else "entire"
        frame.to_excel(writer, sheet_name=f"{split_mode}_{model_name}_{short}"[:31], index=False)

overlap=[]
for split_mode in ("strat_y","strat_y_K"):
    for model_name in ("RF","XGB"):
        t=failure_detail[(split_mode,model_name,"test_data")]
        e=failure_detail[(split_mode,model_name,"entire_data")]
        tk=set(zip(t.dataset.astype(str),t.k_value)); ek=set(zip(e.dataset.astype(str),e.k_value)); shared=tk&ek
        overlap.append({"Stratification":"Target only" if split_mode=="strat_y" else "Target + K",
                        "Model":model_name,"Held-out Important Failures":len(t),
                        "Entire-data Important Failures":len(e),"Same Dataset/K in Both":len(shared),
                        "Held-out Only":len(tk-ek),"Entire-data Only":len(ek-tk)})
important_overlap=pd.DataFrame(overlap)
display(important_overlap)
important_overlap.to_csv(OUTPUT_DIR / "important_failure_test_entire_overlap.csv", index=False)


,Stratification,Model,Held-out Important Failures,Entire-data Important Failures,Same Dataset/K in Both,Held-out Only,Entire-data Only
0,Target only,RF,20,25,20,0,5
1,Target only,XGB,17,17,17,0,0
2,Target + K,RF,20,21,20,0,1
3,Target + K,XGB,20,22,20,0,2


## Slide-ready failure table


In [16]:
slide_failures=failure_summary[failure_summary["Evaluation Scope"].eq("test_data")].copy()
display(slide_failures)
slide_failures.to_csv(OUTPUT_DIR / "slide_table_wrong_and_important_failures.csv", index=False)


,Stratification,Model,Evaluation Scope,Total Predictions,Wrong Predictions,Important Failures,Important / Wrong (%),Important / All (%)
0,Target only,RF,test_data,759,245,20,8.16,2.64
2,Target only,XGB,test_data,759,236,17,7.20,2.24
4,Target + K,RF,test_data,759,262,20,7.63,2.64
6,Target + K,XGB,test_data,759,245,20,8.16,2.64
